# Data Preprocessing — Student Dropout Prediction (FYP)

**Project:** An Explainable Machine Learning Model for Early Prediction of Student Dropout in Secondary Schools

**Dataset:** Student Dropout Analysis and Prediction Dataset (3,000 records, 34 features)

**Objective of this notebook:**  
Prepare the raw student dataset for the Random Forest + SHAP machine learning pipeline. This includes data inspection, handling categorical variables, feature scaling, exploratory data analysis (EDA), and splitting the data into training and testing sets.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries imported successfully.")

## 2. Load the Dataset

The dataset contains 3,000 records of secondary school students with academic, attendance, and behavioral attributes. The original 649 records from the Kaggle Student Dropout Analysis and Prediction Dataset were expanded through statistical data synthesis (adding controlled noise to numerical features) to meet the minimum 3,000 record requirement.

In [ ]:
df = pd.read_csv('student_dropout_3000.csv')

print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nFirst 5 rows:")
df.head()

## 3. Data Inspection

Check the structure, data types, and basic statistics of the dataset.

In [ ]:
print("=" * 60)
print("DATASET INFO")
print("=" * 60)
df.info()

In [ ]:
print("=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)
df.describe()

## 4. Check Missing Values & Duplicates

A clean dataset is critical for reliable model training.

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"After removing duplicates: {df.shape[0]} rows")

## 5. Target Variable Analysis

The target variable `Dropped_Out` indicates whether a student dropped out (At-Risk = True) or not (Not At-Risk = False). This is the label our model will learn to predict.

In [ ]:
target_counts = df['Dropped_Out'].value_counts()
target_pct = df['Dropped_Out'].value_counts(normalize=True) * 100

print("Class Distribution:")
for cls, count in target_counts.items():
    label = 'At-Risk (Dropped Out)' if cls else 'Not At-Risk'
    print(f"  {label}: {count} ({target_pct[cls]:.1f}%)")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

target_counts.plot(kind='bar', ax=ax[0], color=['#2ecc71', '#e74c3c'])
ax[0].set_title('Dropout Class Distribution', fontsize=13, fontweight='bold')
ax[0].set_xlabel('Dropped Out')
ax[0].set_ylabel('Number of Students')
ax[0].set_xticklabels(['Not At-Risk', 'At-Risk'], rotation=0)

ax[1].pie(target_counts, labels=['Not At-Risk', 'At-Risk'],
          autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
ax[1].set_title('Dropout Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

**Observation:** The dataset is imbalanced — approximately 83.9% of students are Not At-Risk while 16.1% are At-Risk. This is realistic for educational data and will be addressed during model training using `class_weight='balanced'` in the Random Forest model and stratified sampling during train-test split. We will evaluate using precision, recall, and F1-score rather than accuracy alone.

## 6. Feature Categorization

Following the project scope, features are grouped into **three main categories** as defined in the FYP proposal:

1. **Academic** — Grades, study time, failures, school/family support, paid classes, higher education aspiration
2. **Attendance** — Number of absences
3. **Behavioral** — Free time, going out, extra-curricular activities, alcohol consumption, health status, relationship status
4. **Demographic (auxiliary)** — Age, gender, family background, parental education, address, guardian

**Why three main categories?** The project scope specifies that the prediction system is based on three main categories: **academic performance, attendance patterns, and behavioral factors**. Demographic features serve as auxiliary context. This categorization aligns with educational research showing that academic, attendance, and behavioral indicators are the strongest early warning signs of student disengagement (Psyridou et al., 2024).

In [ ]:
academic_features = ['Grade_1', 'Grade_2', 'Final_Grade', 'Number_of_Failures',
                     'Study_Time', 'School_Support', 'Family_Support',
                     'Extra_Paid_Class', 'Wants_Higher_Education']

attendance_features = ['Number_of_Absences']

behavioral_features = ['Extra_Curricular_Activities', 'Free_Time', 'Going_Out',
                       'Weekday_Alcohol_Consumption', 'Weekend_Alcohol_Consumption',
                       'Health_Status', 'In_Relationship']

demographic_features = ['School', 'Gender', 'Age', 'Address', 'Family_Size',
                        'Parental_Status', 'Mother_Education', 'Father_Education',
                        'Mother_Job', 'Father_Job', 'Reason_for_Choosing_School',
                        'Guardian', 'Travel_Time', 'Family_Relationship',
                        'Attended_Nursery', 'Internet_Access']

print(f"Academic features:    {len(academic_features)}")
print(f"Attendance features:  {len(attendance_features)}")
print(f"Behavioral features:  {len(behavioral_features)}")
print(f"Demographic features: {len(demographic_features)}")
print(f"Total predictors:     {len(academic_features) + len(attendance_features) + len(behavioral_features) + len(demographic_features)}")

## 7. Exploratory Data Analysis (EDA)

### 7.1 Box Plot Analysis

**What is a Box Plot?**
A box plot (box-and-whisker plot) is a graphical representation that shows the distribution of a numerical variable through five key values:
- **Median** (line inside the box): The middle value of the data
- **Q1 (25th percentile)**: Lower edge of the box — 25% of data falls below this
- **Q3 (75th percentile)**: Upper edge of the box — 75% of data falls below this
- **Whiskers**: Extend to 1.5× the interquartile range (IQR = Q3 - Q1)
- **Outliers**: Individual dots beyond the whiskers, representing extreme values

**Why use box plots here?**
Box plots allow us to visually compare the distribution of key numerical features (grades, absences, failures, study time) between students who dropped out (At-Risk) and those who did not (Not At-Risk). This helps identify which features show the clearest separation between the two groups — features with clear separation are likely to be strong predictors.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(data=df, x='Dropped_Out', y='Final_Grade', ax=axes[0, 0],
            palette=['#2ecc71', '#e74c3c'])
axes[0, 0].set_title('Final Grade vs Dropout Status', fontweight='bold')
axes[0, 0].set_xticklabels(['Not At-Risk', 'At-Risk'])

sns.boxplot(data=df, x='Dropped_Out', y='Number_of_Absences', ax=axes[0, 1],
            palette=['#2ecc71', '#e74c3c'])
axes[0, 1].set_title('Absences vs Dropout Status', fontweight='bold')
axes[0, 1].set_xticklabels(['Not At-Risk', 'At-Risk'])

sns.boxplot(data=df, x='Dropped_Out', y='Number_of_Failures', ax=axes[1, 0],
            palette=['#2ecc71', '#e74c3c'])
axes[1, 0].set_title('Past Failures vs Dropout Status', fontweight='bold')
axes[1, 0].set_xticklabels(['Not At-Risk', 'At-Risk'])

sns.boxplot(data=df, x='Dropped_Out', y='Study_Time', ax=axes[1, 1],
            palette=['#2ecc71', '#e74c3c'])
axes[1, 1].set_title('Study Time vs Dropout Status', fontweight='bold')
axes[1, 1].set_xticklabels(['Not At-Risk', 'At-Risk'])

plt.tight_layout()
plt.show()

**Box Plot Observations:**

1. **Final Grade vs Dropout**: There is a **clear separation** between the two groups. At-Risk students have a significantly lower median Final Grade (around 6-8) compared to Not At-Risk students (around 11-12). The box of At-Risk students barely overlaps with the Not At-Risk group, confirming that **Final Grade is a strong predictor** of dropout.

2. **Absences vs Dropout**: At-Risk students show a **slightly higher median** number of absences and more outliers (extreme absenteeism). However, the overlap between groups is larger, meaning absences alone are a moderate predictor.

3. **Past Failures vs Dropout**: At-Risk students show **more past failures**. The median for Not At-Risk is 0, while At-Risk students have a higher interquartile range. Past academic failure is a significant risk factor.

4. **Study Time vs Dropout**: Both groups have similar distributions, but At-Risk students show a **slightly lower median study time**. Study time has a weaker individual predictive power compared to grades and failures.

### 7.2 Correlation Heatmap

**What is a Correlation Heatmap?**
A heatmap displays the **Pearson correlation coefficient** between each pair of numerical features. The correlation ranges from -1 to +1:
- **+1.0**: Perfect positive correlation — when one goes up, the other always goes up
- **0.0**: No linear correlation — no consistent relationship between the two
- **-1.0**: Perfect negative correlation — when one goes up, the other always goes down

**Why use a heatmap here?**
The heatmap helps us understand:
1. **Which features are correlated with the target (Dropped_Out)** — these are our candidate predictors
2. **Which features are correlated with each other** — highly correlated features (multicollinearity) may provide redundant information
3. **Feature selection decisions** — features with near-zero correlation to the target contribute little predictive value

**Color Interpretation:**
- **Dark Red** (+1.0): Strong positive correlation
- **White** (0.0): No correlation
- **Dark Blue** (-1.0): Strong negative correlation

In [ ]:
# Select numerical columns for correlation analysis
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_corr = df[numerical_cols].copy()
df_corr['Dropped_Out'] = df['Dropped_Out'].astype(int)

plt.figure(figsize=(16, 12))
correlation = df_corr.corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Heatmap Observations:**

1. **Grade Features (Grade_1, Grade_2, Final_Grade)** are **strongly correlated with each other** (0.80+), which is expected — students who perform well in one period tend to perform well in others. All three show **strong negative correlation with Dropped_Out**, meaning higher grades = lower dropout risk.

2. **Number_of_Failures** has a **positive correlation with Dropped_Out** (around 0.30+), confirming that students with more failures are more likely to drop out.

3. **Alcohol Consumption** (Weekend and Weekday) are **positively correlated with each other** (0.60+), and both show a **weak positive correlation with Dropped_Out**.

4. **Age** shows a **slight positive correlation with Dropped_Out** — older students may be more likely to drop out, possibly due to repeated failures or delayed progression.

5. **Study_Time** shows a **weak negative correlation with Dropped_Out** — more study time slightly reduces risk, but the effect is modest.

**Key Insight:** The academic features (Grade_1, Grade_2, Final_Grade, Number_of_Failures) are the strongest predictors of dropout, while behavioral and demographic features provide additional but weaker signals.

In [ ]:
# Feature correlation ranking with Dropout
dropout_corr = df_corr.corr()['Dropped_Out'].drop('Dropped_Out').sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if x > 0 else '#3498db' for x in dropout_corr.values]
dropout_corr.plot(kind='barh', color=colors)
plt.title('Feature Correlation with Dropout (Numerical Features)', fontsize=13, fontweight='bold')
plt.xlabel('Correlation Coefficient')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print("\nTop 10 features correlated with dropout:")
print(dropout_corr.head(10))

## 8. Encode Categorical Variables

Machine learning models require numerical input. Categorical features are converted using:
- **Label Encoding** for binary variables (yes/no, M/F, etc.)
- **One-Hot Encoding** for multi-category nominal features (Mother_Job, Father_Job, Guardian, Reason_for_Choosing_School)

In [ ]:
df_processed = df.copy()

# Binary columns - Label Encoding
binary_cols = ['School', 'Gender', 'Address', 'Family_Size', 'Parental_Status',
               'School_Support', 'Family_Support', 'Extra_Paid_Class',
               'Extra_Curricular_Activities', 'Attended_Nursery',
               'Wants_Higher_Education', 'Internet_Access', 'In_Relationship']

label_encoders = {}
for col in binary_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le

print("Binary categorical features encoded:")
for col in binary_cols:
    classes = label_encoders[col].classes_
    print(f"  {col}: {dict(zip(classes, range(len(classes))))}")

In [ ]:
# Nominal columns - One-Hot Encoding
nominal_cols = ['Mother_Job', 'Father_Job', 'Reason_for_Choosing_School', 'Guardian']

df_processed = pd.get_dummies(df_processed, columns=nominal_cols, drop_first=True)

# Convert dummy columns to int
dummy_cols = [c for c in df_processed.columns if any(c.startswith(n + '_') for n in nominal_cols)]
df_processed[dummy_cols] = df_processed[dummy_cols].astype(int)

# Convert target to int
df_processed['Dropped_Out'] = df_processed['Dropped_Out'].map({True: 1, False: 0, 'True': 1, 'False': 0}).astype(int)

print(f"After one-hot encoding: {df_processed.shape[1]} columns")
df_processed.head()

## 9. Feature Scaling

### Why Feature Scaling?

**StandardScaler** transforms numerical features to have **mean = 0** and **standard deviation = 1**. This is important because:

1. **Different scales**: Features like `Age` (15-22), `Number_of_Absences` (0-93), and `Final_Grade` (0-20) have very different ranges. Without scaling, features with larger ranges would dominate the model.

2. **Equal contribution**: Scaling ensures all features contribute equally during model training, preventing bias toward features with larger numerical values.

3. **SHAP interpretation**: When features are on the same scale, SHAP values become more directly comparable across features.

### Why only scale the 16 numerical features and not all features?

We apply StandardScaler **only to the 16 numerical features** (Age, grades, absences, etc.) and **NOT to the binary/one-hot encoded features** because:

- **Binary features** (0/1) are already on a consistent scale. Scaling them would change their interpretability — a value of 1 means "yes" and 0 means "no". If scaled, these values would become meaningless decimal numbers.

- **One-hot encoded features** (0/1) represent categorical presence/absence. Scaling would distort their meaning and could introduce artifacts.

- **Numerical features** have different ranges and units, so they genuinely need standardization to be comparable.

### Why transform into standardized form (mean=0, std=1)?

The StandardScaler formula is: **z = (x - μ) / σ**

Where x is the original value, μ is the mean, and σ is the standard deviation.

After scaling:
- A value of **0** means the feature is at its average
- A value of **+1** means the feature is 1 standard deviation above average
- A value of **-1** means the feature is 1 standard deviation below average

This allows the model to treat all features fairly and makes the magnitude of SHAP values directly comparable.

In [ ]:
X = df_processed.drop('Dropped_Out', axis=1)
y = df_processed['Dropped_Out']

print(f"Feature matrix X: {X.shape}")
print(f"Target vector y: {y.shape}")

In [ ]:
# Only scale these 16 numerical features
numerical_to_scale = ['Age', 'Mother_Education', 'Father_Education', 'Travel_Time',
                      'Study_Time', 'Number_of_Failures', 'Family_Relationship',
                      'Free_Time', 'Going_Out', 'Weekend_Alcohol_Consumption',
                      'Weekday_Alcohol_Consumption', 'Health_Status',
                      'Number_of_Absences', 'Grade_1', 'Grade_2', 'Final_Grade']

print(f"Features to scale: {len(numerical_to_scale)} out of {X.shape[1]} total")
print(f"Features NOT scaled (binary/one-hot): {X.shape[1] - len(numerical_to_scale)}")

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numerical_to_scale] = scaler.fit_transform(X[numerical_to_scale])

print("\n--- Before Scaling (first 3 numerical features) ---")
print(X[numerical_to_scale[:3]].describe().loc[['mean', 'std', 'min', 'max']].round(2))
print("\n--- After Scaling (mean ≈ 0, std ≈ 1) ---")
print(X_scaled[numerical_to_scale[:3]].describe().loc[['mean', 'std', 'min', 'max']].round(2))

**Scaling Result:** After applying StandardScaler, all 16 numerical features now have mean ≈ 0 and standard deviation ≈ 1. The binary and one-hot encoded features remain unchanged (0 or 1).

## 10. Train-Test Split

Split the dataset into **80% training** and **20% testing**, using **stratified sampling** to preserve the class distribution in both sets — important due to the class imbalance (83.9% Not At-Risk vs 16.1% At-Risk).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")
print(f"\nClass distribution in training set:")
print(y_train.value_counts(normalize=True).round(3))
print(f"\nClass distribution in testing set:")
print(y_test.value_counts(normalize=True).round(3))

## 11. Save Preprocessed Data

The cleaned and preprocessed datasets are saved as CSV files for use in the next phase: **Random Forest model training and SHAP explainability**.

In [ ]:
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Preprocessed data saved successfully:")
print(f"  - X_train.csv ({X_train.shape[0]} samples, {X_train.shape[1]} features)")
print(f"  - X_test.csv ({X_test.shape[0]} samples, {X_test.shape[1]} features)")
print(f"  - y_train.csv ({y_train.shape[0]} labels)")
print(f"  - y_test.csv ({y_test.shape[0]} labels)")

## 12. Summary

| Step | Task | Details | Status |
|------|------|---------|--------|
| 1 | Import libraries | pandas, numpy, sklearn, seaborn, matplotlib | ✅ Done |
| 2 | Load dataset | 3,000 records, 34 features | ✅ Done |
| 3 | Data inspection | Data types, statistics | ✅ Done |
| 4 | Missing values & duplicates | 0 missing, duplicates removed | ✅ Done |
| 5 | Target analysis | 83.9% Not At-Risk, 16.1% At-Risk | ✅ Done |
| 6 | Feature categorization | Academic(9), Attendance(1), Behavioral(7), Demographic(16) | ✅ Done |
| 7 | EDA — Box plots | Compared distributions by dropout status | ✅ Done |
| 7 | EDA — Heatmap | Correlation analysis of all numerical features | ✅ Done |
| 8 | Categorical encoding | Label + One-Hot encoding | ✅ Done |
| 9 | Feature scaling | StandardScaler on 16 numerical features only | ✅ Done |
| 10 | Train-test split | 80/20, stratified | ✅ Done |
| 11 | Save preprocessed data | X_train, X_test, y_train, y_test | ✅ Done |

**Next Step:** Model Training with Random Forest + SHAP Explainability